In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from google import genai

load_dotenv()

gemini_client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [3]:
def llm(prompt):
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

In [4]:
llm("Hey, What's up?")

"Hey there! Not much on my end, just here and ready to help! What's up with you today?\n\nHow can I assist you? Feel free to ask me anything!"

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

That's great news! It's exciting to discover a course that interests you.

To give you a definitive answer, I'll need a little more information about **which specific course you're referring to.**

Please tell me:

1.  **What is the name of the course?**
2.  **Where did you discover it?** (e.g., Coursera, Udemy, edX, a university website, a specific platform, a private offering, etc.)

Once I have that information, I can often check the enrollment status for you or direct you to where you can find out if you can join now, if there's a waiting list, or if the next enrollment period is coming up.

Generally speaking:
*   **Self-paced courses** (like many on platforms such as Coursera, Udemy) often allow you to join at any time.
*   **Cohort-based courses** or those with specific start dates might have fixed enrollment periods, and you might need to wait for the next session if the current one has already started or closed for enrollment.

Looking forward to helping you find out!


In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students partici

In [9]:
question = "I just discovered the course. Can I join now?"
answer = llm(prompt)
print(answer)

Yes, you can join now. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted.


1. Retrieval

In [10]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [11]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [12]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [13]:
documents[1300]

{'id': 'c1095e75a3',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 10. Kubernetes and TensorFlow Serving',
 'question': "Module 10 / Apple Silicon: tensorflow/serving:2.7.0 exits with 'Illegal instruction'",
 'answer': "The official `tensorflow/serving` image isn't built for arm64. Two options:\n\n1. Update Docker Desktop to ≥ 4.35 and enable **Docker VMM (Beta)** which uses Rosetta to emulate amd64 — the official image then runs.\n2. Use the `bitnami/tensorflow-serving:2` image. Slightly different config: model files go under `/bitnami/model-data/1/`, and the env var becomes `TENSORFLOW_SERVING_MODEL_NAME` instead of `MODEL_NAME`."}

In [14]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [15]:
search_results = index.search(
    question, 
    boost_dict={"question": 2.0, "answer": 1.0, "section": 1.0},
    filter_dict={"course": "llm-zoomcamp"}, 
    num_results=5
    )

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [16]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [17]:
search_results = search(question)

In [18]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

2. Building The Prompt (Augmentation)

In [19]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [20]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [21]:
# The context is a formatted string with all the search results

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [22]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [23]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [24]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

3. LLM (Generation)

In [25]:
response = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

In [31]:
response.text

'Yes, you can join and start learning now. You don\'t need to register; you can just start learning and submitting homework (while the form is open).\n\nHowever, if your goal is to receive a certificate, there are important conditions:\n*   You need to submit your project while submissions are still being accepted.\n*   Certificates are only awarded if you finish the course with a "live" cohort, not in self-paced mode. This is because you must peer-review other capstone projects, which only occurs when the course is actively running and the peer-review list is compiled.\n\nSo, while you can always start learning, make sure you check if the submission period for the current cohort\'s project is still open if you\'re aiming for a certificate. If not, you would need to wait for the next official offering (e.g., Summer 2025). Missing homework is generally not an issue for certificates, as long as you pass the Capstone project.'

In [35]:
response = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config={
        "system_instruction": INSTRUCTIONS
    }
)

In [36]:
response.text

'Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [37]:
def llm(instructions, user_prompt, model="gemini-2.5-flash"):
    response = gemini_client.models.generate_content(
        model=model,
        contents=user_prompt,
        config={
            "system_instruction": instructions
        }
    )

    return response.text

Full RAG Pipeline

In [38]:
def rag(query, model="gemini-2.5-flash"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [39]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join now. You can start learning and submitting homework. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [40]:
rag("How do I get a certificate?")

'To get a certificate, you need to:\n\n1.  **Finish the course with a "live" cohort.** Certificates are not awarded for the self-paced mode.\n2.  **Pass the Capstone project.** This is mandatory.\n3.  **Peer-review 3 capstone projects** after submitting your own. Peer-reviewing can only be done while the course is running.\n4.  **Ensure your official name** (as in your identification documents) is entered in the "second field" of your course profile. This name will appear on your certificate.\n\nMissing homework will not prevent you from getting a certificate, as long as you pass the Capstone project.'